# NTI sweep: guidance forward × invert methods

Для каждого `*.mp3` в `real_music` и для каждого метода из `[gci, uni_euler, uni_guidance_euler]` запускается `edit_music.py` с `stepper=guidance_euler`, NTI и теми же overrides, что в твоём shell-примере.

- **gci:** `continuation_steps`, `invert_stepper.guidance_scale`, `j_approx`, `j_eps`
- **uni_guidance_euler:** только `invert_stepper.guidance_scale`
- **uni_euler:** без дополнительных `invert_stepper.*`

Запуск из корня репо обычно не нужен: в ячейке ниже `REPO_ROOT` = родитель `notebooks/`. Папка с треками: сначала `<repo>/real_music`, иначе `<repo>/../real_music`.

Поставь `DRY_RUN = True`, чтобы только напечатать команды.

In [ ]:
from __future__ import annotations

import os
import subprocess
import sys
from pathlib import Path

# --- настройки ---
DRY_RUN = True  # True: только print команды
CONTINUE_ON_ERROR = True  # не останавливаться на первом падении
USE_CAFFEINATE = sys.platform == "darwin"

INV_METHODS = ("gci", "uni_euler", "uni_guidance_euler")
GS = 2.0
INFERENCE_STEPS = 50
GCI_CONTINUATION_STEPS = 10
GCI_J_EPS = "1.0"

REPO_ROOT = Path("..").resolve()
assert (REPO_ROOT / "edit_music.py").is_file(), f"edit_music.py не найден: {REPO_ROOT}"


def hydra_music_dir_rel(repo_root: Path, music_dir: Path) -> str:
    rr, d = repo_root.resolve(), music_dir.resolve()
    try:
        return d.relative_to(rr).as_posix()
    except ValueError:
        return Path(os.path.relpath(d, rr)).as_posix()


def resolve_real_music(repo_root: Path) -> tuple[Path, str] | None:
    rr = repo_root.resolve()
    for d in (rr / "real_music", rr.parent / "real_music"):
        if d.is_dir():
            return d, hydra_music_dir_rel(rr, d)
    return None


def track_prompt_id(mp3: Path) -> str:
    return mp3.stem.replace("-", "_")


def build_hydra_args(
    *,
    inv_method: str,
    track_id: str,
    mp3_name: str,
    music_dir_rel: str,
) -> list[str]:
    exp = f"__NTI___track_{track_id}___method_{inv_method}___gs_{GS}"
    music_path = f"{music_dir_rel}/{mp3_name}"
    args: list[str] = [
        f"stepper@invert_stepper={inv_method}",
        "stepper=guidance_euler",
        f"prompt@p2p_task.src=detailed/{track_id}",
        f"prompt@p2p_task.tgt=detailed/gender/{track_id}",
        f"exp_name={exp}",
        f"music_path={music_path}",
        "log_local_error=true",
        f"inference_steps={INFERENCE_STEPS}",
        f"stepper.guidance_scale={GS}",
        "log_trajectory_images=true",
        "nti.enabled=true",
        "nti.epsilon=1.0e-8",
        "nti.num_inner_steps=10",
        "nti.optimize_first_outer_steps=20",
    ]
    if inv_method == "gci":
        args += [
            f"invert_stepper.continuation_steps={GCI_CONTINUATION_STEPS}",
            f"invert_stepper.guidance_scale={GS}",
            "invert_stepper.j_approx=true",
            f"invert_stepper.j_eps={GCI_J_EPS}",
        ]
    elif inv_method == "uni_guidance_euler":
        args.append(f"invert_stepper.guidance_scale={GS}")
    return args


def run_one(hydra_args: list[str]) -> int:
    cmd = [sys.executable, str(REPO_ROOT / "edit_music.py"), *hydra_args]
    if USE_CAFFEINATE:
        cmd = ["caffeinate", "-i", *cmd]
    line = subprocess.list2cmdline(cmd)
    print(line, flush=True)
    if DRY_RUN:
        return 0
    return subprocess.run(cmd, cwd=REPO_ROOT).returncode


resolved = resolve_real_music(REPO_ROOT)
if resolved is None:
    raise FileNotFoundError(
        f"Нет папки real_music: проверь {REPO_ROOT / 'real_music'} или {REPO_ROOT.parent / 'real_music'}"
    )
real_music, music_dir_rel = resolved
mp3s = sorted(real_music.glob("*.mp3"))
if not mp3s:
    raise FileNotFoundError(f"В {real_music} нет *.mp3")

print(f"REPO_ROOT={REPO_ROOT}")
print(f"real_music={real_music}  (music_path префикс: {music_dir_rel}/)")
print(f"треки: {[p.name for p in mp3s]}")
print(f"методы: {INV_METHODS}")
print(f"DRY_RUN={DRY_RUN} USE_CAFFEINATE={USE_CAFFEINATE}")
print("---")

failures: list[str] = []
for mp3 in mp3s:
    tid = track_prompt_id(mp3)
    for method in INV_METHODS:
        hydra = build_hydra_args(
            inv_method=method,
            track_id=tid,
            mp3_name=mp3.name,
            music_dir_rel=music_dir_rel,
        )
        rc = run_one(hydra)
        if rc != 0:
            msg = f"rc={rc} track={tid} method={method}"
            print(msg, file=sys.stderr)
            failures.append(msg)
            if not CONTINUE_ON_ERROR:
                raise SystemExit(rc)

if failures:
    print(f"\nОшибок: {len(failures)}", file=sys.stderr)
    for m in failures:
        print(m, file=sys.stderr)
else:
    print("Готово: все запуски завершились с кодом 0.")